# Bickley jet

In [1]:
# # An unstable Bickley jet in Shallow Water model
#
# This example uses Oceananigans.jl's `ShallowWaterModel` to simulate
# the evolution of an unstable, geostrophically balanced, Bickley jet
# The example is periodic in ``x`` with flat bathymetry and
# uses the conservative formulation of the shallow water equations.
# The initial conditions superpose the Bickley jet with small-amplitude perturbations.
# See ["The nonlinear evolution of barotropically unstable jets," J. Phys. Oceanogr. (2003)](https://doi.org/10.1175/1520-0485(2003)033<2173:TNEOBU>2.0.CO;2)
# for more details on this problem.
#
# The mass transport ``(uh, vh)`` is the prognostic momentum variable
# in the conservative formulation of the shallow water equations,
# where ``(u, v)`` are the horizontal velocity components and ``h``
# is the layer height.
#
# ## Install dependencies
#
# First we make sure that we have all of the packages that are required to
# run the simulation.
#
# ```julia
# using Pkg
# pkg"add Oceananigans, NCDatasets, Polynomials, CairoMakie"
# ```

using Oceananigans
using Oceananigans.Models: ShallowWaterModel

# ## Two-dimensional domain
#
# The shallow water model is two-dimensional and uses grids that are `Flat`
# in the vertical direction. We use length scales non-dimensionalized by the width
# of the Bickley jet.

grid = RectilinearGrid(size = (128, 256),
                       x = (0, 2π),
                       y = (-2π, 2π),
                       topology = (Periodic, Bounded, Flat))

# ## Building a `ShallowWaterModel`
#
# We build a `ShallowWaterModel` with the `WENO` advection scheme,
# 3rd-order Runge-Kutta time-stepping, non-dimensional Coriolis, and
# gravitational acceleration

gravitational_acceleration = 1
coriolis = FPlane(f=1)

model = ShallowWaterModel(; grid, coriolis, gravitational_acceleration,
                          timestepper = :RungeKutta3,
                          momentum_advection = WENO())

# ## Background state and perturbation
#
# The background velocity ``ū`` and free-surface ``η̄`` correspond to a
# geostrophically balanced Bickely jet with maximum speed of ``U`` and maximum
# free-surface deformation of ``Δη``,

U = 1  # Maximum jet velocity
H = 10 # Reference depth
f = coriolis.f
g = gravitational_acceleration
Δη = f * U / g  # Maximum free-surface deformation as dictated by geostrophy

h̄(x, y) = H - Δη * tanh(y)
ū(x, y) = U * sech(y)^2

# The total height of the fluid is ``h = L_z + \eta``. Linear stability theory predicts that
# for the parameters we consider here, the growth rate for the most unstable mode that fits
# our domain is approximately ``0.139``.

# The vorticity of the background state is

ω̄(x, y) = 2 * U * sech(y)^2 * tanh(y)

# The initial conditions include a small-amplitude perturbation that decays away from the
# center of the jet.

small_amplitude = 1e-4
k = 1 # meridional decay rate of initial random perturbations

uⁱ(x, y) = ū(x, y) + small_amplitude * exp(-k*y^2) * randn()
uhⁱ(x, y) = uⁱ(x, y) * h̄(x, y)

# We first set a "clean" initial condition without noise for the purpose of discretely
# calculating the initial 'mean' vorticity,

ū̄h(x, y) = ū(x, y) * h̄(x, y)

set!(model, uh = ū̄h, h = h̄)

# We next compute the initial vorticity and perturbation vorticity,

uh, vh, h = model.solution

## Build velocities
u = uh / h
v = vh / h

## Build mean vorticity discretely
ω = Field(∂x(v) - ∂y(u))

## Copy mean vorticity to a new field
ωⁱ = Field((Face, Face, Nothing), model.grid)
ωⁱ .= ω

## Use this new field to compute the perturbation vorticity
ω′ = Field(ω - ωⁱ)

# and finally set the "true" initial condition with noise,

set!(model, uh = uhⁱ)

# ## Running a `Simulation`
#
# We pick the time-step so that we make sure we resolve the surface gravity waves, which
# propagate with speed of the order ``\sqrt{g H}``. That is, with `Δt = 1e-2` we ensure
# that `` \sqrt{g H} Δt / Δx,  \sqrt{g H} Δt / Δy < 0.7``.

simulation = Simulation(model, Δt = 5e-3, stop_time = 150);

┌ Warning: The ShallowWaterModel is currently unvalidated, subject to change, and should not be used for scientific research without adequate validation.
└ @ Oceananigans.Models.ShallowWaterModels ~/.julia/packages/Oceananigans/Rb6LJ/src/Models/ShallowWaterModels/shallow_water_model.jl:129


In [2]:
# --------------------
# Output writer (NetCDF): write u, v, ω every 0.5 time units
# `write_grid=true` puts grid coordinates in the file,
# which your plotting cell reads as x_* / y_* arrays.
# --------------------
using NCDatasets

u_field = uh / h
v_field = vh / h
ω_field = Field(∂x(v_field) - ∂y(u_field))

exp_name = string("shallow_water_Bickley_jet_k=",k)
simulation.output_writers[:fields] = NetCDFWriter(
    model,
    (; u = u_field, v = v_field, ω = ω_field),
    schedule = TimeInterval(0.5),
    filename = string("../data/raw_simulation_output/", exp_name, ".nc"),
    overwrite_existing = true,
)

# --------------------
# Run
# --------------------
run!(simulation)

┌ Warning: Overwriting existing /Users/henrifdrake/code/ESS280-gfd/data/raw_simulation_output/shallow_water_Bickley_jet_k=1.nc.
└ @ OceananigansNCDatasetsExt ~/.julia/packages/Oceananigans/Rb6LJ/ext/OceananigansNCDatasetsExt.jl:1023
[ Info: Initializing simulation...
[ Info:     ... simulation initialization complete (3.574 seconds)
[ Info: Executing initial time step...
[ Info:     ... initial time step complete (2.228 seconds).
[ Info: Simulation is stopping after running for 0 seconds.
[ Info: Simulation time 2.500 minutes equals or exceeds stop time 2.500 minutes.


In [3]:
# ## Visualize the results

using Printf, CairoMakie
nothing #hide

# Read the 2-D output and build a 2×2 panel figure:
#   Row 1: total u, perturbation u
#   Row 2: total ω, perturbation ω
fig = Figure(size = (1200, 800))

axis_kwargs = (xlabel = "x", ylabel = "y")

# Row 1: u
ax_u   = Axis(fig[1, 1]; title = "Zonal velocity, u",                    axis_kwargs...)
ax_u′  = Axis(fig[1, 3]; title = "Perturbation u - u₀ (or ū)",           axis_kwargs...)

# Row 2: ω
ax_ω   = Axis(fig[2, 1]; title = "Total vorticity, ω",                   axis_kwargs...)
ax_ω′  = Axis(fig[2, 3]; title = "Perturbation ω - ω₀ (or ω̄)",          axis_kwargs...)

n = Observable(1)

ds = NCDataset(simulation.output_writers[:fields].filepath, "r")

times = ds["time"][:]

# Coordinates (swap to u/v-point coords if your grid is staggered)
x = ds["x_faa"];  y = ds["y_afa"]

# --- Total fields (reactive) ---
u_plot = @lift ds["u"][:, :, $n]
ω_plot = @lift ds["ω"][:, :, $n]

# --- Reference fields for perturbations (e.g., initial snapshot; replace with means if desired) ---
ui = ds["u"][:, :, 1]
ωi = ds["ω"][:, :, 1]

# --- Reactive perturbations ---
upert = @lift $u_plot .- ui
ωpert = @lift $ω_plot .- ωi

# --- Robust, symmetric, adaptive color ranges (ignore NaNs/Infs; avoid (0,0) collapse) ---
colorrange_sym = A -> @lift begin
    B = $A
    maxabs = mapreduce(x -> isfinite(x) ? abs(x) : 0, max, B; init = 0.0)
    maxabs = (!isfinite(maxabs) || maxabs == 0) ? 1f-9 : maxabs
    (-maxabs, maxabs)
end

cr_u′ = colorrange_sym(upert)
cr_ω′ = colorrange_sym(ωpert)

# --- Plots ---
# Total u (let Makie autorange)
hm_u  = heatmap!(ax_u,  x, y, u_plot; colormap = :balance)
Colorbar(fig[1, 2], hm_u)

# Perturbation u with adaptive symmetric range
hm_u′ = heatmap!(ax_u′, x, y, upert; colorrange = cr_u′, colormap = :balance)
Colorbar(fig[1, 4], hm_u′)

# Total ω (keep your fixed range)
hm_ω  = heatmap!(ax_ω,  x, y, ω_plot; colorrange = (-1, 1), colormap = :balance)
Colorbar(fig[2, 2], hm_ω)

# Perturbation ω with adaptive symmetric range
hm_ω′ = heatmap!(ax_ω′, x, y, ωpert; colorrange = cr_ω′, colormap = :balance)
Colorbar(fig[2, 4], hm_ω′)

# Title banner row
title = @lift @sprintf("t = %.1f", times[$n])
fig[0, 1:4] = Label(fig, title, fontsize=24, tellwidth=false)

current_figure() #hide
fig

# --- Record movie ---
frames = 1:length(times)
record(fig, string("../movies/", exp_name, ".mp4"), frames, framerate=12) do i
    n[] = i
end
nothing #hide

# Close the NetCDF when done
close(ds)

closed Dataset